# ViPRA — Qwen2.5-Math-7B, search cấu hình và retrain riêng

Backbone **7,62B**, LoRA FP16 + **FSDP2 song song trên T4 x2**. Bật Internet và Add Input chứa hai CSV.
Clone nhánh `Nhotest` vào `/kaggle/working/VLSP`:

```bash
git clone --branch Nhotest --single-branch https://github.com/NgocNhoNguyen2605n/VLSP.git /kaggle/working/VLSP
```

Chỉ chia train 90/10. Giữ câu trùng cùng phía và chấm 5 epoch trên cùng validation.
Chọn best epoch theo đáp án + đơn vị trên validation rồi dừng. Thử các cấu hình với RUN_DIR riêng.
Chỉ khi chọn cấu hình thắng mới retrain từ backbone gốc trên train + validation. Dev dành cho đánh giá cuối.


In [ ]:
import os  # Thiết lập môi trường cho các tiến trình train/evaluate.
import subprocess  # Chạy Python riêng để trả lại VRAM sau mỗi tác vụ.
import sys  # Dùng đúng Python của notebook Kaggle.
from pathlib import Path  # Đường dẫn repo, Kaggle Input và kết quả.

REPO = Path("/kaggle/working/VLSP")
DATA_ROOT = "/kaggle/input"
RUN_DIR = "/kaggle/working/vipra_qwen25_math_7b_5epochs"  # Chọn thư mục mới cho mỗi experiment.
assert (REPO / "requirements.txt").is_file(), "Hãy clone repo vào /kaggle/working/VLSP trước."

os.environ.update(PYTHONHASHSEED="42", CUBLAS_WORKSPACE_CONFIG=":4096:8",
                  TOKENIZERS_PARALLELISM="false", HF_HUB_DISABLE_PROGRESS_BARS="1",
                  HF_HOME="/tmp/vipra_hf_cache")  # Không lưu backbone vào output notebook.

def run(module, *options):
    "Chạy tác vụ trong repo, dừng notebook nếu có lỗi."
    subprocess.run([sys.executable, "-m", module, *options], cwd=REPO, check=True)

# Gỡ torchao cũ của Kaggle: PEFT kiểm tra cả khi chỉ dùng LoRA FP16.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y",
                "torchao"], check=True)

# Cài dependency; cần PyTorch CUDA >= 2.8 cho FSDP2.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(REPO / "requirements.txt")], check=True)


## 1. Kiểm tra dữ liệu và template

Tự tìm CSV trong Input. Nếu có nhiều bản cùng tên, thu hẹp `DATA_ROOT`
hoặc truyền `--train-file` và `--dev-file` cụ thể vào cả check/train.
Không tải trọng số backbone trong bước này.


In [ ]:
# Kiểm tra các mẫu và mask assistant; mẫu dài sẽ báo lỗi thay vì bị cắt.
run("baseline.check", "--data-root", DATA_ROOT)


## 2. Search 5 epoch và chọn best epoch (chưa retrain)

Một run với **seed 42**, train đủ **5 epoch**.
676 mẫu có nhãn được chia thành **608 train / 68 validation** bằng split seed 42.
Sau mỗi epoch, sinh greedy theo batch 4 trên validation và lưu checkpoint + predictions + metrics.
Chọn content_tolerant accuracy cao nhất; nếu hòa: exact, loss, epoch sớm.

Sau khi chọn cấu hình thắng, cell retrain riêng bên dưới nạp lại backbone gốc và tạo LoRA mới trên đủ 676 mẫu (train + validation).
Giữ seed 42, train đúng số epoch tốt nhất (ví dụ best epoch 3 thì retrain 3 epoch).
Optimizer/scheduler được tạo mới. Cấu hình ở `configs/baseline.json`: batch 4/GPU, tích lũy 2 bước/GPU x 2 GPU = effective batch 16, max_length 2048.

Mỗi run tự dùng torchrun tạo hai worker FSDP2, hai T4 học các batch riêng đồng thời. SDPA math giữ trung gian FP32.
Search và retrain chạy tuần tự, mỗi run có hai worker; VRAM được trả trước khi nạp lại backbone.
Validation cần cả hai rank cùng tham gia generation; chỉ rank 0 ghi điểm. Có thể mất nhiều thời gian. Log có seed/epoch và tiến độ validation.

`RUN_DIR` là thư mục experiment mới. Nếu run thất bại, giữ log và chọn RUN_DIR mới.
Gradient overflow: cả hai rank cùng bỏ bước và giảm scale; quá nhiều overflow liên tiếp hoặc loss NaN/Inf sẽ dừng.

Batch được gom theo độ dài để giảm padding. Activation checkpoint nằm bên trong FSDP; tắt backward prefetch, mỗi decoder chỉ all-gather khi backward cần lớp đó.
Log cập nhật một dòng batch/ETA của epoch; `train_log.json` có thời gian training từng epoch. NCCL timeout sau 5 phút nếu collective bị kẹt.
Nếu lần train đầu hiện `676 train / 0 validation`, kiểm tra commit và cấu hình training_stage. Bản mới tự chuyển baseline.train CLI mặc định sang experiment; cell baseline.experiment bên dưới vẫn là cách gọi trực tiếp.
Sau NCCL abort, restart session Kaggle, cập nhật repo và chọn thư mục output mới trước khi chạy lại.

Tiến độ: `Epoch 1/5 | Batch 11/76 | Còn epoch ~00:07:35`, cập nhật trên cùng dòng. Validation loss và sinh đáp án có ETA riêng; mỗi batch generation có tối đa 4 câu, 68 câu = 17 batch/epoch. Giới hạn vẫn 300 token/câu. Search 76 batch/epoch, retrain 85 batch/epoch. Chỉ rank 0 in. `verbose_logging=false` ẩn thông tin chi tiết; cấu hình/loss/metrics vẫn lưu trong JSON.

FSDP chỉ shard backbone frozen; LoRA FP32 replicate và all-reduce sau khi backward hoàn tất, không reduce-scatter gradient. Trước mỗi run có preflight CUDA/NCCL bằng Qwen nhỏ: phải hiện `Preflight PASS` rồi mới nạp 7B. Mỗi run lưu `rank-0-progress.json` và `rank-1-progress.json` để biết pha cuối của từng GPU khi bị lỗi.

Notebook cũ gọi `baseline.train` rồi `baseline.evaluate --run-dir RUN_DIR` cũng được hỗ trợ: train mặc định chỉ search; gọi --retrain-from sau khi chọn cấu hình, evaluate tự tìm `RUN_DIR/final` khi retrain hoàn tất.

Search hoàn tất lưu selection.json rồi dừng. Mỗi cấu hình dùng RUN_DIR mới và cùng split seed.


In [ ]:
# Search: split -> train 5 epoch -> chọn best epoch -> dừng, không retrain.
run("baseline.experiment", "--data-root", DATA_ROOT, "--output-dir", RUN_DIR)


## 3. Retrain riêng cấu hình thắng

So sánh các selection.json trên cùng split, rồi đặt SELECTED_RUN_DIR thành thư mục search đã chọn.
Mặc định None: Run All sẽ không retrain. Cấu hình được lấy từ experiment đã lưu, không từ baseline.json hiện tại.
Backbone gốc + LoRA mới, toàn bộ 676 mẫu, số epoch của checkpoint thắng.


In [ ]:
SELECTED_RUN_DIR = None  # Ví dụ: "/kaggle/working/trial_lr_1e4" khi đã chọn cấu hình thắng.
if SELECTED_RUN_DIR is None:
    print("Chưa chọn cấu hình; bỏ qua retrain.")
else:
    run("baseline.experiment", "--retrain-from", SELECTED_RUN_DIR)


## 4. Đánh giá adapter cuối trên dev

Dùng `SELECTED_RUN_DIR/final/adapter`, sau khi retrain cấu hình đã chọn hoàn tất.
Dev không dùng chọn checkpoint. Điểm validation search không phải điểm độc lập
của adapter cuối vì validation đã được gộp vào train cuối.

P1 ước lượng yêu cầu đáp án/đơn vị đúng và thời gian không quá 60 giây.
Dung sai tạm `rtol=0.01`, `atol=0`. Sinh greedy một lần mỗi câu, lưu raw output và thời gian.


In [ ]:
# Chỉ đánh giá sau khi chủ động chọn và retrain cấu hình thắng.
if SELECTED_RUN_DIR is None:
    print("Chưa chọn cấu hình để retrain; bỏ qua dev.")
else:
    FINAL_DIR = str(Path(SELECTED_RUN_DIR) / "final")
    assert (Path(FINAL_DIR) / "adapter" / "adapter_config.json").is_file(), "Hãy retrain cấu hình đã chọn trước."
    run("baseline.evaluate", "--run-dir", FINAL_DIR)


Split: `RUN_DIR/split.json`. Xếp hạng: `RUN_DIR/selection.json`.
Run tìm epoch: `RUN_DIR/search/`; adapter cuối: `SELECTED_RUN_DIR/final/adapter/`.
Điểm dev: `SELECTED_RUN_DIR/final/evaluation/metrics.json` và `predictions.jsonl`.
Muốn đánh giá lại cùng adapter, truyền `--result-dir` khác để không ghi đè.
